# Viability Data Merge

This notebook merges viability assay data:

## Data Sources
1. **merged_caspase_mito_hepg2.csv**: Already merged HepG2 viability data (AID 720634 + AID 1346981)
2. **AID 1346979**: CHO-K1 viability counter screen

## Merge Logic
- Remove rows with missing CID
- **Keep CID with different outcome values** (Active, Inactive, Inconclusive)
- **If same CID + same outcome**: Keep the record with **highest activity score**

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

print("Libraries loaded successfully!")

Libraries loaded successfully!


## Configuration: Viability Assay Metadata

In [2]:
# Viability Assay Metadata
VIABILITY_METADATA = {
    '720634': {
        'assay_name': 'AID_720634_cell_viability',
        'assay_type': 'viability',
        'cell_line': 'HepG2',
        'source': 'Tox21',
        'is_counter_screen': True,
        'biological_process': 'cytotoxicity',
        'endpoint_level': 'general_toxicity'
    },
    '1346981': {
        'assay_name': 'AID_1346981_viability_HepG2',
        'assay_type': 'viability',
        'cell_line': 'HepG2',
        'source': 'Tox21',
        'is_counter_screen': True,
        'biological_process': 'cytotoxicity',
        'endpoint_level': 'general_toxicity'
    },
    '1346979': {
        'assay_name': 'AID_1346979_viability_CHO-K1',
        'assay_type': 'viability',
        'cell_line': 'CHO-K1',
        'source': 'Tox21',
        'is_counter_screen': True,
        'biological_process': 'cytotoxicity',
        'endpoint_level': 'general_toxicity'
    },
}

# Standard output columns
STANDARD_COLS = [
    'CID', 'SMILES', 'outcome', 'activity_score', 'AID', 'assay_name', 
    'assay_type', 'cell_line', 'source', 'is_counter_screen', 
    'biological_process', 'endpoint_level'
]

print(f"Defined metadata for {len(VIABILITY_METADATA)} viability assays.")

Defined metadata for 3 viability assays.


---
## Section 1: Load Existing HepG2 Viability Data

In [3]:
# Load the already merged HepG2 viability data
hepg2_path = 'merged_caspase_mito_hepg2.csv'

hepg2_df = pd.read_csv(hepg2_path)

print("Loaded merged_caspase_mito_hepg2.csv")
print(f"Shape: {hepg2_df.shape}")
print(f"Columns: {hepg2_df.columns.tolist()}")
print(f"\nUnique CIDs: {hepg2_df['CID'].nunique():,}")
print(f"\nOutcome distribution:")
print(hepg2_df['outcome'].value_counts())
print(f"\nCell lines: {hepg2_df['cell_line'].unique()}")

Loaded merged_caspase_mito_hepg2.csv
Shape: (10431, 11)
Columns: ['CID', 'SMILES', 'outcome', 'score', 'assay_name', 'assay_type', 'cell_line', 'source', 'is_counter_screen', 'biological_process', 'endpoint_level']

Unique CIDs: 8,734

Outcome distribution:
outcome
Inactive        8200
Inconclusive    1477
Active           754
Name: count, dtype: int64

Cell lines: ['HepG2']


In [4]:
# Standardize column names to match expected output
# The existing file uses 'score' instead of 'activity_score'
hepg2_df = hepg2_df.rename(columns={'score': 'activity_score'})

# Add AID column if not present (extract from assay_name)
if 'AID' not in hepg2_df.columns:
    hepg2_df['AID'] = hepg2_df['assay_name'].str.extract(r'AID_(\d+)')[0]

print("Standardized HepG2 data columns:")
print(hepg2_df.columns.tolist())
print(f"\nFirst 5 rows:")
display(hepg2_df.head())

Standardized HepG2 data columns:
['CID', 'SMILES', 'outcome', 'activity_score', 'assay_name', 'assay_type', 'cell_line', 'source', 'is_counter_screen', 'biological_process', 'endpoint_level', 'AID']

First 5 rows:


,CID,SMILES,outcome,activity_score,assay_name,assay_type,cell_line,source,is_counter_screen,biological_process,endpoint_level,AID
0,4,CC(CN)O,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity,720634
1,6,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,Active,81,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity,720634
2,11,C(CCl)Cl,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity,720634
3,13,C1=CC(=C(C=C1Cl)Cl)Cl,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity,720634
4,33,C(C=O)Cl,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity,720634


---
## Section 2: Load and Clean AID 1346979 (CHO-K1 Viability)

In [5]:
# Load AID 1346979 - CHO-K1 viability counter screen
chok1_path = 'Data/tox21_caspase/AID_1346979_datatable_counter_screen.csv'

# Skip metadata rows (rows 2-6)
chok1_raw = pd.read_csv(chok1_path, skiprows=[1, 2, 3, 4, 5], low_memory=False)

print("AID 1346979 - CHO-K1 Viability")
print(f"Loaded shape: {chok1_raw.shape}")

# Clean CID column
chok1_raw['PUBCHEM_CID'] = pd.to_numeric(chok1_raw['PUBCHEM_CID'], errors='coerce')
missing_cid = chok1_raw['PUBCHEM_CID'].isna().sum()
print(f"Rows with missing CID: {missing_cid}")

chok1_raw = chok1_raw.dropna(subset=['PUBCHEM_CID'])
chok1_raw['PUBCHEM_CID'] = chok1_raw['PUBCHEM_CID'].astype(int)

# Clean activity score
chok1_raw['PUBCHEM_ACTIVITY_SCORE'] = pd.to_numeric(chok1_raw['PUBCHEM_ACTIVITY_SCORE'], errors='coerce').fillna(0)

print(f"After cleanup: {chok1_raw.shape}")
print(f"Unique CIDs: {chok1_raw['PUBCHEM_CID'].nunique():,}")
print(f"\nOutcome distribution:")
print(chok1_raw['PUBCHEM_ACTIVITY_OUTCOME'].value_counts())

AID 1346979 - CHO-K1 Viability
Loaded shape: (9667, 2049)
Rows with missing CID: 143
After cleanup: (9524, 2049)
Unique CIDs: 7,671

Outcome distribution:
PUBCHEM_ACTIVITY_OUTCOME
Inactive        6893
Active          1377
Inconclusive    1254
Name: count, dtype: int64


In [6]:
# Standardize CHO-K1 data to match expected columns
metadata = VIABILITY_METADATA['1346979']

chok1_std = pd.DataFrame({
    'CID': chok1_raw['PUBCHEM_CID'].values,
    'SMILES': chok1_raw['PUBCHEM_EXT_DATASOURCE_SMILES'].values,
    'outcome': chok1_raw['PUBCHEM_ACTIVITY_OUTCOME'].values,
    'activity_score': chok1_raw['PUBCHEM_ACTIVITY_SCORE'].values,
    'AID': '1346979',
    'assay_name': metadata['assay_name'],
    'assay_type': metadata['assay_type'],
    'cell_line': metadata['cell_line'],
    'source': metadata['source'],
    'is_counter_screen': metadata['is_counter_screen'],
    'biological_process': metadata['biological_process'],
    'endpoint_level': metadata['endpoint_level']
})

print(f"Standardized CHO-K1 data:")
print(f"Shape: {chok1_std.shape}")
print(f"Columns: {chok1_std.columns.tolist()}")
display(chok1_std.head())

Standardized CHO-K1 data:
Shape: (9524, 12)
Columns: ['CID', 'SMILES', 'outcome', 'activity_score', 'AID', 'assay_name', 'assay_type', 'cell_line', 'source', 'is_counter_screen', 'biological_process', 'endpoint_level']


,CID,SMILES,outcome,activity_score,AID,assay_name,assay_type,cell_line,source,is_counter_screen,biological_process,endpoint_level
0,17931,CN1C(=S)CN=C(C2=C1C=CC(=C2)Cl)C3=CC=CC=C3,Inactive,0,1346979,AID_1346979_viability_CHO-K1,viability,CHO-K1,Tox21,True,cytotoxicity,general_toxicity
1,3045407,CO[C@H]1[C@@H](C[C@@H]2CN3CCC4=C([C@H]3C[C@@H]...,Inactive,0,1346979,AID_1346979_viability_CHO-K1,viability,CHO-K1,Tox21,True,cytotoxicity,general_toxicity
2,14708,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@]2(C)O)CC[C@@...,Active,81,1346979,AID_1346979_viability_CHO-K1,viability,CHO-K1,Tox21,True,cytotoxicity,general_toxicity
3,3085168,CC1=NC=C(C(=N1)N)CN(C=O)/C(=C(\CCO)/SS/C(=C(/C...,Inactive,0,1346979,AID_1346979_viability_CHO-K1,viability,CHO-K1,Tox21,True,cytotoxicity,general_toxicity
4,54158,CC1=CC(=C(C(=C1NC(=O)CN(CC(=O)O)CC(=O)O)C)Br)C,Inactive,0,1346979,AID_1346979_viability_CHO-K1,viability,CHO-K1,Tox21,True,cytotoxicity,general_toxicity


---
## Section 3: Combine All Viability Data

In [7]:
# Ensure both dataframes have the same columns
print("Checking column alignment...")
print(f"HepG2 columns: {set(hepg2_df.columns)}")
print(f"CHO-K1 columns: {set(chok1_std.columns)}")

# Select only standard columns from both
hepg2_subset = hepg2_df[[c for c in STANDARD_COLS if c in hepg2_df.columns]].copy()
chok1_subset = chok1_std[[c for c in STANDARD_COLS if c in chok1_std.columns]].copy()

# Combine
combined_raw = pd.concat([hepg2_subset, chok1_subset], ignore_index=True)

print(f"\nCombined dataset: {len(combined_raw):,} rows")
print(f"Unique CIDs: {combined_raw['CID'].nunique():,}")
print(f"\nBy source:")
print(combined_raw['source'].value_counts())
print(f"\nBy cell line:")
print(combined_raw['cell_line'].value_counts())

Checking column alignment...
HepG2 columns: {'CID', 'endpoint_level', 'assay_name', 'source', 'outcome', 'is_counter_screen', 'cell_line', 'assay_type', 'AID', 'biological_process', 'activity_score', 'SMILES'}
CHO-K1 columns: {'CID', 'endpoint_level', 'assay_name', 'source', 'outcome', 'is_counter_screen', 'cell_line', 'assay_type', 'AID', 'biological_process', 'activity_score', 'SMILES'}

Combined dataset: 19,955 rows
Unique CIDs: 8,734

By source:
source
Tox21    19955
Name: count, dtype: int64

By cell line:
cell_line
HepG2     10431
CHO-K1     9524
Name: count, dtype: int64


---
## Section 4: Deduplication

**Logic:**
1. If same CID has **different outcomes** → Keep ALL
2. If same CID + same outcome but **different cell lines** → Keep ALL
3. If same CID + same outcome + same cell line → Keep the one with **highest activity score**

In [ ]:
print("Deduplication Process")
print("="*70)
print("Logic:")
print("  - Keep CID with DIFFERENT outcome values (keep all)")
print("  - Keep CID with SAME outcome but DIFFERENT cell lines (keep all)")
print("  - If same CID + same outcome + same cell_line: Keep HIGHEST activity score")
print()

print(f"Before deduplication: {len(combined_raw):,} rows")

# Sort by CID, outcome, cell_line, score (descending)
combined_sorted = combined_raw.sort_values(
    by=['CID', 'outcome', 'cell_line', 'activity_score'],
    ascending=[True, True, True, False]
)

# Keep first occurrence of each CID + outcome + cell_line (highest score)
merged_viability = combined_sorted.drop_duplicates(
    subset=['CID', 'outcome', 'cell_line'],
    keep='first'
).reset_index(drop=True)

print(f"After deduplication: {len(merged_viability):,} rows")
print(f"Removed: {len(combined_raw) - len(merged_viability):,} duplicate rows")
print(f"\nUnique CIDs: {merged_viability['CID'].nunique():,}")

In [ ]:
# Verify deduplication logic
print("Verification: CIDs with multiple records")
print("-"*50)

# Count by CID + outcome combinations
cid_outcome_counts = merged_viability.groupby('CID')['outcome'].nunique()
print("CIDs by number of distinct outcomes:")
print(f"  1 outcome: {(cid_outcome_counts == 1).sum():,}")
print(f"  2 outcomes: {(cid_outcome_counts == 2).sum():,}")
print(f"  3 outcomes: {(cid_outcome_counts == 3).sum():,}")

# Count CIDs that appear in both cell lines
cid_cellline_counts = merged_viability.groupby('CID')['cell_line'].nunique()
print(f"\nCIDs by number of cell lines:")
print(f"  1 cell line: {(cid_cellline_counts == 1).sum():,}")
print(f"  2 cell lines: {(cid_cellline_counts == 2).sum():,}")

# Show example of CID with same outcome in different cell lines
cids_both_celllines = cid_cellline_counts[cid_cellline_counts == 2].index
if len(cids_both_celllines) > 0:
    # Find a CID that has same outcome in both cell lines
    for example_cid in cids_both_celllines[:20]:
        cid_data = merged_viability[merged_viability['CID'] == example_cid]
        if cid_data['outcome'].duplicated().any():
            print(f"\nExample - CID {example_cid} with same outcome in different cell lines:")
            display(cid_data[['CID', 'outcome', 'activity_score', 'cell_line', 'assay_name']])
            break

---
## Section 5: Final Summary and Export

In [10]:
print("="*70)
print("FINAL MERGED VIABILITY DATASET SUMMARY")
print("="*70)

print(f"\nTotal records: {len(merged_viability):,}")
print(f"Unique CIDs: {merged_viability['CID'].nunique():,}")

print(f"\n--- By Activity Outcome ---")
for outcome, count in merged_viability['outcome'].value_counts().items():
    pct = count / len(merged_viability) * 100
    print(f"  {outcome}: {count:,} ({pct:.1f}%)")

print(f"\n--- By Cell Line ---")
cell_summary = merged_viability.groupby('cell_line').agg({
    'CID': ['count', 'nunique']
})
cell_summary.columns = ['Total Records', 'Unique CIDs']
print(cell_summary.sort_values('Total Records', ascending=False))

print(f"\n--- By Assay ---")
print(merged_viability['assay_name'].value_counts())

FINAL MERGED VIABILITY DATASET SUMMARY

Total records: 11,609
Unique CIDs: 8,734

--- By Activity Outcome ---
  Inactive: 8,213 (70.7%)
  Inconclusive: 2,123 (18.3%)
  Active: 1,273 (11.0%)

--- By Cell Line ---
           Total Records  Unique CIDs
cell_line                            
HepG2              10114         8651
CHO-K1              1495         1482

--- By Assay ---
assay_name
AID_720634_mito_HepG2           8186
AID_1346981_caspase37_HepG2     1928
AID_1346979_viability_CHO-K1    1495
Name: count, dtype: int64


In [11]:
# Preview
print("First 20 rows:")
display(merged_viability.head(20))

First 20 rows:


,CID,SMILES,outcome,activity_score,AID,assay_name,assay_type,cell_line,source,is_counter_screen,biological_process,endpoint_level
0,4,CC(CN)O,Inactive,0,720634,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
1,6,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,Active,83,1346979,AID_1346979_viability_CHO-K1,viability,CHO-K1,Tox21,True,cytotoxicity,general_toxicity
2,11,C(CCl)Cl,Inactive,0,720634,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
3,13,C1=CC(=C(C=C1Cl)Cl)Cl,Inactive,0,720634,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
4,33,C(C=O)Cl,Inactive,0,720634,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
5,34,C(CCl)O,Inactive,0,720634,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
6,51,C(CC(=O)O)C(=O)C(=O)O,Inactive,0,720634,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
7,66,C1=CC(=C(C=C1Cl)O)Cl,Inactive,0,720634,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
8,66,C1=CC(=C(C=C1Cl)O)Cl,Inconclusive,20,720634,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
9,72,C1=CC(=C(C=C1C(=O)O)O)O,Inactive,0,720634,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity


In [ ]:
# Save merged viability data
output_path = 'output/merged_all_viability_data.csv'
os.makedirs('output', exist_ok=True)

merged_viability.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Shape: {merged_viability.shape}")
print(f"Columns: {merged_viability.columns.tolist()}")

---
## Summary

This notebook merged:
- **merged_caspase_mito_hepg2.csv**: HepG2 viability data (AID 720634 + AID 1346981)
- **AID 1346979**: CHO-K1 viability counter screen

**Merge Logic Applied:**
1. Removed rows with missing CID
2. Kept CID with different outcome values (Active, Inactive, Inconclusive)
3. Kept CID with same outcome but different cell lines
4. For same CID + same outcome + same cell_line: Kept record with highest activity score

**Output:** `output/merged_all_viability_data.csv`